# Parametric particle-trajectory PINN
One-way analytical Poiseuille flow with linear Stokes drag. The network learns a position correction from acceleration residuals. Start this notebook from the repository root or from the package directory.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import sys

root = Path.cwd()
if root.name == "parametric_particle_pinn":
    root = root.parent
sys.path.insert(0, str(root))
from parametric_particle_pinn.model import read_config, load_checkpoint, relaxation_time
from parametric_particle_pinn.train import train

config = read_config()
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = root / "particle-pinn-runs" / stamp
print("Stokes relaxation time:", relaxation_time(config))
print("Output:", run_dir)

## Train using particle physics
Adam fits the transverse head and then the horizontal head. Horizontal L-BFGS reuses one fixed collocation bank. Development residuals select the best checkpoint.

In [ ]:
training_summary = train(config, run_dir, device="cpu")
training_summary

## Load one checkpoint and predict several releases
This cell needs only the saved model and inference API. The releases below are independent of the training and validation banks. Query positions, velocities and accelerations before the predicted stop event.

In [ ]:
import numpy as np
from parametric_particle_pinn.inference import predict_trajectory, wall_release

checkpoint = run_dir / "best.pt"
model = load_checkpoint(checkpoint)
releases = np.vstack((
    wall_release(config, "lower", 0.439, -0.18, 0.41),
    wall_release(config, "upper", 0.751, 0.83, 0.27),
    [0.684, 0.12, 0.29, 0.0],
    [1.531, 0.228, -0.17, 0.65],
))
times = np.linspace(0, config["domain"]["horizon"], 101)
prediction = predict_trajectory(model, *releases.T, times)
for release, event, stop, mask in zip(releases, prediction["event"]["type"],
                                       prediction["event"]["time"], prediction["valid"]):
    print(release, event, round(stop, 6), "valid samples:", int(mask.sum()))

## Independent final validation
The ODE is used only here. Accuracy and event criteria were fixed in the configuration before final validation.

In [ ]:
from parametric_particle_pinn.validate import validate

report = validate(checkpoint, run_dir / "validation", split="final")
print("Accepted:", report["passed_count"], "/", report["case_count"])
from IPython.display import Image, display
for image in ("trajectories.png", "errors.png"):
    display(Image(filename=str(run_dir / "validation" / image)))
assert report["accepted"], report["failed_cases"]